# PA-S1. Начало работы: продуктовая аналитика H&M

**Дисциплина:** «Продуктовая аналитика», МГТУ им. Н.Э. Баумана, осень 2026
**Трек:** Б (индивидуальный)
**Автор:** _ФИО_ (вклад: вся работа — постановка, проверка данных, расчёты, выводы)
**Репозиторий:** _ссылка на репозиторий_

**Как запустить.** Клонировать репозиторий, установить зависимости из `requirements.txt`, положить в папку `data/` три файла соревнования (или запустить `scripts/download_data.py` с токеном Kaggle) и выполнить ноутбук сверху вниз. Все пути относительные, корень — папка репозитория. Полный прогон на 31,8 млн строк занимает около 5–10 минут; при первом запуске данные один раз загружаются в локальную базу DuckDB `data/hm.duckdb`.

**Содержание**
1. Продукт
2. Данные: источник, лицензия, объём, проверка качества
3. Определения
4. Метрики (пирамида)
5. План трекинга
6. Первые расчёты: когортный retention и LTV/CAC
7. Этика
8. Критерии успеха
9. Ограничения и обоснование синтетических данных

## 0. Настройка окружения

Параметры, от которых зависят определения и расчёты, собраны в одной ячейке. Они фиксируются здесь и не меняются в следующих сдачах.

In [ ]:
import sys, subprocess, platform
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}" if float(x).is_integer() else f"{x:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})

DATA_DIR = Path("data")      # исходные файлы соревнования (в git не входят)
OUT_DIR = Path("outputs")    # сгенерированные таблицы (входят в git)
OUT_DIR.mkdir(exist_ok=True)

FILES = {"transactions": "transactions_train.csv",
         "customers": "customers.csv",
         "articles": "articles.csv"}

# --- зафиксированные параметры определений (раздел 3) ---
PRICE_SCALE = 1000            # цена нормирована; 1 у.е. = price * 1000
ACTIVE_WINDOW_DAYS = 28       # активный покупатель: покупка за последние 28 дней
RETURN_LOOKBACK_DAYS = 365    # повторный покупатель месяца: покупка за 365 дней до начала месяца
CONVERSION_DAYS = 90          # конверсия: вторая покупка (другой день) за 90 дней после первой
CHURN_DAYS = 90               # отток: нет покупок 90 дней
COHORT_START = "2019-09-01"   # когорты: месяцы первой покупки сентябрь 2019 - февраль 2020
COHORT_END = "2020-03-01"     # (до этого идёт «прогрев» истории, после - мало месяцев наблюдения)
MAX_K = 6                     # горизонт когорт и LTV: месяцы 0..6
TEST_ITEMS_PER_DAY = 100      # тестовый / нерозничный покупатель: >= 100 позиций в один день
SEED = 42

missing = [f for f in FILES.values() if not (DATA_DIR / f).exists()]
if missing:
    print("Нет файлов:", missing, "- скачиваю через Kaggle API")
    subprocess.run([sys.executable, "scripts/download_data.py"], check=True)

print("Python", platform.python_version(), "| duckdb", duckdb.__version__, "| pandas", pd.__version__,
      "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)
for f in FILES.values():
    print(f"{f:<24} {(DATA_DIR / f).stat().st_size / 1e6:>9,.1f} MB")

In [ ]:
# Загрузка в локальную базу DuckDB (один раз). article_id читаем строкой: у него ведущий ноль.
con = duckdb.connect(str(DATA_DIR / "hm.duckdb"))
con.execute("SET preserve_insertion_order = false")
existing = set(con.sql("SELECT table_name FROM information_schema.tables").df()["table_name"])

def p(name):
    return (DATA_DIR / FILES[name]).as_posix()

if "tx" not in existing:
    con.execute(f'''
        CREATE TABLE tx AS
        SELECT t_dat, customer_id, lpad(article_id, 10, '0') AS article_id, price, sales_channel_id
        FROM read_csv('{p("transactions")}', header = true,
             columns = {{'t_dat': 'DATE', 'customer_id': 'VARCHAR', 'article_id': 'VARCHAR',
                         'price': 'DOUBLE', 'sales_channel_id': 'INTEGER'}})''')
if "customers" not in existing:
    con.execute(f'''
        CREATE TABLE customers AS
        SELECT * FROM read_csv('{p("customers")}', header = true,
             types = {{'customer_id': 'VARCHAR', 'age': 'DOUBLE', 'postal_code': 'VARCHAR',
                       'club_member_status': 'VARCHAR', 'fashion_news_frequency': 'VARCHAR'}})''')
if "articles" not in existing:
    con.execute(f'''
        CREATE TABLE articles AS
        SELECT * REPLACE (lpad(article_id, 10, '0') AS article_id)
        FROM read_csv('{p("articles")}', header = true,
             types = {{'article_id': 'VARCHAR', 'product_code': 'VARCHAR'}})''')

for t in ["tx", "customers", "articles"]:
    n = con.sql(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<10} {n:>12,} строк")

## 1. Продукт

**Что за продукт.** Омниканальная fashion-розница H&M Group: интернет-магазин и офлайн-магазины. Компания работает на 53 онлайн-рынках и имеет около 4 850 магазинов (описание соревнования на Kaggle). Ассортимент огромный: в выгрузке 105 542 артикула.

**Для кого.** Массовый покупатель одежды. По данным `customers.csv` основная аудитория 20–35 лет (распределение возраста показано в разделе 2), заметная доля участников клубной программы.

**Какую проблему решает продуктовая команда.** При таком ассортименте покупатель не находит нужное быстро, не покупает или не возвращается за следующей покупкой. В данных это видно прямо: из новых покупателей только около 12–16 % покупают снова в следующем месяце (раздел 6).

**Гипотеза ценности.** Если показывать новому покупателю персональные рекомендации после первой покупки, он быстрее найдёт следующую вещь, и доля новых покупателей, купивших повторно за 90 дней, вырастет. Это увеличит число повторных покупателей (North Star) и LTV, не увеличивая расходы на привлечение.

**Бизнес-модель.** Выручка от продажи товаров; валовая маржа H&M Group за 2019 финансовый год 52,6 % (годовой отчёт). Покупатели приходят через маркетинг (расходы на привлечение, CAC) и покупают через два канала продаж. Экономика юнита: окупается ли привлечение покупателя его валовой прибылью за первые месяцы (LTV/CAC).

**Роль аналитики в проекте.** На исторических транзакциях описать путь покупателя (первая покупка → повторные → отток), выбрать метрики, проверить, где теряются покупатели, и подготовить базу для оценки продуктовых изменений в следующих сдачах.

## 2. Данные

| Параметр | Значение |
|---|---|
| Источник | Kaggle, соревнование *H&M Personalized Fashion Recommendations* (H&M Group, 2022) |
| Файлы | `transactions_train.csv` (3,3 ГБ), `customers.csv` (198 МБ), `articles.csv` (36 МБ); картинки не используются |
| Лицензия | Правила соревнования Kaggle (вкладка *Rules*): данные для участия в соревновании и некоммерческого / академического использования, без распространения. Поэтому **данные не лежат в репозитории**, их скачивает `scripts/download_data.py` |
| Версия | У данных соревнования нет версий; файлы не менялись с публикации (январь 2022). Контроль: число строк и диапазон дат ниже |
| Период | 2018-09-20 — 2020-09-22 (734 дня) |
| Объём | 31,8 млн строк покупок, 1,37 млн покупателей, 105,5 тыс. артикулов |
| Тип событий | только покупки (строка = купленная позиция), дата без времени |

**Структура.** `transactions_train`: дата, покупатель, артикул, цена (нормирована, без валюты), канал продаж (1 или 2; расшифровка в описании не дана). `customers`: возраст, статус клуба, подписка на новости, хеш почтового индекса. `articles`: иерархия товара (21 группа одежды, 19 групп продуктов), цвет, описание.

In [ ]:
ov = con.sql('''
    SELECT COUNT(*) AS rows, COUNT(DISTINCT customer_id) AS customers, COUNT(DISTINCT article_id) AS articles,
           MIN(t_dat) AS date_min, MAX(t_dat) AS date_max, COUNT(DISTINCT t_dat) AS days_with_data,
           date_diff('day', MIN(t_dat), MAX(t_dat)) + 1 AS days_in_range,
           AVG(CASE WHEN sales_channel_id = 2 THEN 1.0 ELSE 0 END) AS share_channel_2,
           MIN(price) AS price_min, quantile_cont(price, 0.5) AS price_p50,
           quantile_cont(price, 0.99) AS price_p99, MAX(price) AS price_max
    FROM tx''').df().T
ov.columns = ["значение"]
print(ov.to_string())
MAX_DATE = pd.Timestamp(con.sql("SELECT MAX(t_dat) FROM tx").fetchone()[0])
MIN_DATE = pd.Timestamp(con.sql("SELECT MIN(t_dat) FROM tx").fetchone()[0])

### 2.1 Пропуски

Доля пустых значений по всем столбцам трёх таблиц. В ключевых полях покупок (дата, покупатель, артикул, цена, канал) пропусков быть не должно.

In [ ]:
def null_shares(table):
    cols = con.sql(f"DESCRIBE {table}").df()["column_name"].tolist()
    expr = ", ".join(f'AVG(CASE WHEN "{c}" IS NULL THEN 1.0 ELSE 0 END) AS "{c}"' for c in cols)
    s = con.sql(f"SELECT {expr} FROM {table}").df().T[0]
    return s[s > 0].sort_values(ascending=False)

for t in ["tx", "customers", "articles"]:
    s = null_shares(t)
    print(f"\n{t}: столбцов с пропусками {len(s)}")
    if len(s):
        print((s * 100).round(2).astype(str).add(" %").to_string())

### 2.2 Дубли

В `transactions_train` нет номера чека и количества. Если покупатель купил две одинаковые вещи в один день, это две одинаковые строки. По описанию соревнования это нормальные покупки нескольких штук, а не дубли выгрузки, поэтому мы их **не удаляем** (иначе занизим выручку). Дубли ключей в справочниках проверяем отдельно.

In [ ]:
dup = con.sql('''
    SELECT (SELECT COUNT(*) FROM tx) AS rows,
           (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM tx)) AS distinct_rows,
           (SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM customers) AS dup_customer_id,
           (SELECT COUNT(*) - COUNT(DISTINCT article_id) FROM articles) AS dup_article_id,
           (SELECT COUNT(*) FROM (SELECT DISTINCT customer_id FROM tx)
             WHERE customer_id NOT IN (SELECT customer_id FROM customers)) AS tx_customers_not_in_table,
           (SELECT COUNT(*) FROM tx WHERE article_id NOT IN (SELECT article_id FROM articles)) AS tx_rows_unknown_article
''').df().T[0]
dup["repeated_rows"] = dup["rows"] - dup["distinct_rows"]
dup["repeated_share_%"] = dup["repeated_rows"] / dup["rows"] * 100
print(dup.to_string())

### 2.3 Согласованность значений

Проверяем категориальные поля и заглушки: разные написания одного значения, почтовый индекс-заглушка, возраст вне правдоподобного диапазона, неизвестный тип товара.

In [ ]:
print(con.sql('''SELECT COALESCE(fashion_news_frequency, '<пусто>') AS fashion_news_frequency, COUNT(*) AS n FROM customers GROUP BY 1 ORDER BY 2 DESC''').df().to_string(index=False))
print()
print(con.sql('''SELECT COALESCE(club_member_status, '<пусто>') AS club_member_status, COUNT(*) AS n FROM customers GROUP BY 1 ORDER BY 2 DESC''').df().to_string(index=False))
print()
top_postal = con.sql('''SELECT postal_code, COUNT(*) AS n FROM customers GROUP BY 1 ORDER BY 2 DESC LIMIT 3''').df()
top_postal["postal_code"] = top_postal["postal_code"].str[:12] + "..."
print(top_postal.to_string(index=False))
print()
print(con.sql('''
    SELECT MIN(age) AS age_min, MAX(age) AS age_max,
           AVG(CASE WHEN age < 16 OR age > 99 THEN 1.0 ELSE 0 END) AS age_out_of_range,
           quantile_cont(age, [0.25, 0.5, 0.75]) AS age_quartiles
    FROM customers''').df().to_string(index=False))
print()
print(con.sql('''
    SELECT COUNT(*) AS articles,
           AVG(CASE WHEN product_type_no = -1 THEN 1.0 ELSE 0 END) AS product_type_unknown,
           AVG(CASE WHEN garment_group_name = 'Unknown' THEN 1.0 ELSE 0 END) AS garment_group_unknown
    FROM articles''').df().to_string(index=False))

In [ ]:
ages = con.sql("SELECT age FROM customers WHERE age IS NOT NULL").df()["age"]
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(ages, bins=range(15, 101), color="#4C72B0")
ax.set_title("Возраст покупателей (customers.csv)")
ax.set_xlabel("возраст, лет"); ax.set_ylabel("покупателей")
plt.tight_layout(); plt.show()

### 2.4 Выбросы

Цена нормирована, поэтому абсолютные пороги не имеют смысла. Используем два правила:
- **межквартильный размах логарифма цены** (общий выброс по всей выборке);
- **отклонение от медианной цены того же артикула** больше чем в 5 раз в любую сторону. Это похоже на ошибку ввода: один и тот же товар не может стоить в 5 раз дороже или дешевле обычного (скидки H&M обычно до 50–70 %).

In [ ]:
q1, q3 = con.sql("SELECT quantile_cont(ln(price), 0.25), quantile_cont(ln(price), 0.75) FROM tx").fetchone()
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
out = con.sql(f'''
    WITH med AS (SELECT article_id, median(price) AS m FROM tx GROUP BY 1)
    SELECT AVG(CASE WHEN ln(price) < {lo} OR ln(price) > {hi} THEN 1.0 ELSE 0 END) AS iqr_outlier_share,
           AVG(CASE WHEN price > 5 * m OR price < m / 5 THEN 1.0 ELSE 0 END) AS article_price_anomaly_share,
           SUM(CASE WHEN price > 5 * m OR price < m / 5 THEN 1 ELSE 0 END) AS article_price_anomaly_rows
    FROM tx JOIN med USING (article_id)''').df().T[0]
print(f"Границы IQR по цене: [{np.exp(lo):.5f}; {np.exp(hi):.5f}]")
print(out.to_string())

### 2.5 Тестовые и нерозничные покупатели

Явных тестовых записей (ID вида `test`, нулевые цены, покупатели без карточки) в данных нет, это проверено выше. Ищем поведенческие аномалии: покупатель, который берёт 100 и больше позиций в один день, похож на оптового закупщика, сотрудника или тестовый аккаунт, а не на розничного покупателя. Таких покупателей исключаем из расчёта продуктовых метрик (но не из проверки качества).

Для дальнейших расчётов строим таблицу **заказов**: заказ = все покупки одного покупателя в один день (номера чека в данных нет).

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE orders_all AS
    SELECT customer_id, t_dat, COUNT(*) AS items, SUM(price) AS rev,
           CASE WHEN SUM(CASE WHEN sales_channel_id = 2 THEN price ELSE 0 END)
                   > SUM(CASE WHEN sales_channel_id = 1 THEN price ELSE 0 END) THEN 2 ELSE 1 END AS ch
    FROM tx GROUP BY 1, 2''')
con.execute(f'''
    CREATE OR REPLACE TABLE test_users AS
    SELECT customer_id, MAX(items) AS max_items_per_day
    FROM orders_all GROUP BY 1 HAVING MAX(items) >= {TEST_ITEMS_PER_DAY}''')
con.execute('''
    CREATE OR REPLACE VIEW orders AS
    SELECT * FROM orders_all WHERE customer_id NOT IN (SELECT customer_id FROM test_users)''')

print(con.sql('''
    SELECT COUNT(*) AS orders, AVG(items) AS items_mean, quantile_cont(items, 0.99) AS items_p99,
           quantile_cont(items, 0.9999) AS items_p9999, MAX(items) AS items_max
    FROM orders_all''').df().T[0].to_string())
tu = con.sql('''
    SELECT COUNT(*) AS test_users,
           (SELECT COUNT(*) FROM tx WHERE customer_id IN (SELECT customer_id FROM test_users)) AS their_rows
    FROM test_users''').df().T[0]
print(tu.to_string())

### 2.6 Границы времени

- Данные начинаются 2018-09-20, поэтому у покупателей, которые пришли раньше, первая покупка «обрезана»: в первые месяцы как будто появляется очень много новых покупателей. Это **прогрев истории**, когорты строим только с сентября 2019 года, когда за каждым покупателем есть минимум год истории.
- Данные заканчиваются во вторник 2020-09-22, сентябрь 2020 неполный. Месячные метрики считаем только по полным месяцам.
- Для правил с окном вперёд (конверсия и отток за 90 дней) берём только тех, у кого после события прошло 90 дней наблюдения.
- Март–май 2020: закрытие магазинов из-за пандемии. Это структурный сдвиг, его нужно учитывать при сравнении периодов.

In [ ]:
newm = con.sql('''
    WITH f AS (SELECT customer_id, MIN(t_dat) AS first_d FROM orders GROUP BY 1)
    SELECT CAST(date_trunc('month', first_d) AS DATE) AS month, COUNT(*) AS new_customers
    FROM f GROUP BY 1 ORDER BY 1''').df()
gap = con.sql('''
    SELECT date_diff('day', MIN(t_dat), MAX(t_dat)) + 1 - COUNT(DISTINCT t_dat) FROM tx''').fetchone()[0]
warm = con.sql(f'''
    WITH f AS (SELECT customer_id, MIN(t_dat) AS first_d FROM orders GROUP BY 1)
    SELECT AVG(CASE WHEN first_d < DATE '{MIN_DATE.date()}' + INTERVAL 90 DAY THEN 1.0 ELSE 0 END) FROM f''').fetchone()[0]
print(f"Дней без данных внутри периода: {gap}")
print(f"Последний день: {MAX_DATE.date()} ({MAX_DATE.day_name()})")
print(f"Доля покупателей с первой покупкой в первые 90 дней данных: {warm:.1%}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(newm["month"], newm["new_customers"], width=20, color="#4C72B0")
ax.axvspan(pd.Timestamp(COHORT_START), pd.Timestamp(COHORT_END), color="#DD8452", alpha=0.2,
           label="когорты для анализа")
ax.set_title("Покупатели по месяцу первой покупки: прогрев истории в 2018 году")
ax.set_ylabel("новых покупателей"); ax.legend()
plt.tight_layout(); plt.show()

### 2.7 Сводка дефектов и решения

Порог задания: дефектов до 5 %. Синтетические дефекты **не добавляем**: естественных хватает, чтобы показать очистку, а их доля в ключевых полях ниже порога.

In [ ]:
def share(sql):
    return con.sql(sql).fetchone()[0]

defects = pd.DataFrame([
    ("customers", "age пустой", share("SELECT AVG(CASE WHEN age IS NULL THEN 1.0 ELSE 0 END) FROM customers"),
     "не используем возраст в этих срезах, отдельная категория «нет данных»"),
    ("customers", "club_member_status пустой", share("SELECT AVG(CASE WHEN club_member_status IS NULL THEN 1.0 ELSE 0 END) FROM customers"),
     "категория «нет данных»"),
    ("customers", "fashion_news_frequency пустой", share("SELECT AVG(CASE WHEN fashion_news_frequency IS NULL THEN 1.0 ELSE 0 END) FROM customers"),
     "категория «нет данных»"),
    ("customers", "fashion_news_frequency: 'None' и 'NONE'", share("SELECT AVG(CASE WHEN fashion_news_frequency = 'None' THEN 1.0 ELSE 0 END) FROM customers"),
     "приводим к одному значению NONE"),
    ("customers", "postal_code заглушка (один хеш)", share("SELECT MAX(n) * 1.0 / SUM(n) FROM (SELECT COUNT(*) n FROM customers GROUP BY postal_code)"),
     "не ключевое поле; гео-срез не строим"),
    ("articles", "garment_group = Unknown", share("SELECT AVG(CASE WHEN garment_group_name = 'Unknown' THEN 1.0 ELSE 0 END) FROM articles"),
     "отдельная группа, средняя маржа"),
    ("articles", "product_type_no = -1", share("SELECT AVG(CASE WHEN product_type_no = -1 THEN 1.0 ELSE 0 END) FROM articles"),
     "оставляем, на метрики не влияет"),
    ("tx", "ключевые поля пустые", share('''SELECT AVG(CASE WHEN t_dat IS NULL OR customer_id IS NULL OR article_id IS NULL
                                              OR price IS NULL OR sales_channel_id IS NULL THEN 1.0 ELSE 0 END) FROM tx'''),
     "—"),
    ("tx", "товар неизвестного типа", share('''SELECT AVG(CASE WHEN a.product_type_no = -1 THEN 1.0 ELSE 0 END)
                                              FROM tx JOIN articles a USING (article_id)'''),
     "оставляем"),
    ("tx", "цена в 5+ раз от медианы артикула", out["article_price_anomaly_share"],
     "оставляем в выручке, проверить невозможно"),
    ("tx", "строки тестовых/оптовых покупателей", tu["their_rows"] / ov.loc["rows", "значение"],
     "исключаем из метрик"),
], columns=["таблица", "дефект", "доля", "решение"])
defects["доля, %"] = (defects.pop("доля") * 100).round(2)
print(defects[["таблица", "дефект", "доля, %", "решение"]].to_string(index=False, float_format=lambda x: f"{x:.2f}"))

tx_any = share(f'''
    WITH med AS (SELECT article_id, median(price) AS m FROM tx GROUP BY 1)
    SELECT AVG(CASE WHEN a.product_type_no = -1 OR price > 5 * m OR price < m / 5
                     OR customer_id IN (SELECT customer_id FROM test_users) THEN 1.0 ELSE 0 END)
    FROM tx JOIN med USING (article_id) JOIN articles a USING (article_id)''')
print(f"\nСтроки покупок хотя бы с одним дефектом: {tx_any:.2%} (порог задания - до 5 %)")

**Почему датасет подходит.** Различимый покупатель (`customer_id`) есть у каждой покупки, поэтому строятся когорты, LTV и сегменты. Окно 734 дня с двумя сезонами, 1,36 млн покупателей. Отток определяется по датам покупок, платежи есть (цена каждой позиции). Данные реальные, не «игрушечные»: скошенные распределения, выбросы цены, естественные пропуски и несогласованные значения, которые видны выше. Чего нет и что это значит, перечислено в разделе 9.

## 3. Определения

Фиксируются здесь и не меняются в следующих сдачах. Числовые параметры заданы в ячейке настройки (раздел 0).

| Понятие | Определение |
|---|---|
| Покупатель | `customer_id`, кроме тестовых/оптовых (≥ 100 позиций в один день) |
| Заказ | все покупки покупателя в один календарный день (номера чека нет) |
| Канал покупателя | канал первого заказа; канал заказа = канал с большей суммой в этот день (при равенстве канал 1) |
| Дата появления | дата первого заказа в данных (даты регистрации нет) |
| **Активный покупатель** | сделал хотя бы один заказ за последние 28 дней; месячная версия — хотя бы один заказ в календарном месяце |
| Повторный покупатель месяца | активен в месяце и делал заказ в течение 365 дней до начала месяца |
| **Конверсия** | доля новых покупателей, сделавших второй заказ (в другой день) в течение 90 дней после первого. Воронки просмотров в данных нет, поэтому конверсия считается в повторную покупку |
| **Когорта** | календарный месяц первого заказа; анализируются когорты сентябрь 2019 — февраль 2020 |
| **Удержание (retention) месяца k** | доля когорты, сделавшая хотя бы один заказ в календарном месяце «месяц когорты + k», k = 0…6 |
| **Правило оттока** | покупатель ушёл, если после его последнего заказа прошло больше 90 дней (на конец данных или после конца месяца для месячной метрики). Считается только для тех, за кем есть 90 дней наблюдения |
| Выручка, у.е. | `price × 1000` (цены нормированы, валюты нет) |
| LTV (месяцы 0–6) | накопленная валовая прибыль на одного покупателя когорты к концу месяца k |

In [ ]:
# Таблица покупателей: всё, что нужно для когорт, конверсии и оттока, по одному определению.
con.execute('''
    CREATE OR REPLACE TABLE cust AS
    WITH o AS (SELECT *, row_number() OVER (PARTITION BY customer_id ORDER BY t_dat) AS n FROM orders)
    SELECT o.customer_id,
           MIN(t_dat) AS first_d, MAX(t_dat) AS last_d, COUNT(*) AS n_orders,
           MAX(CASE WHEN n = 2 THEN t_dat END) AS second_d,
           MAX(CASE WHEN n = 1 THEN ch END) AS first_ch,
           CAST(date_trunc('month', MIN(t_dat)) AS DATE) AS cohort
    FROM o GROUP BY 1''')

churn = con.sql(f'''
    SELECT COUNT(*) AS eligible,
           SUM(CASE WHEN last_d < DATE '{MAX_DATE.date()}' - INTERVAL {CHURN_DAYS} DAY THEN 1 ELSE 0 END) AS churned
    FROM cust WHERE first_d <= DATE '{MAX_DATE.date()}' - INTERVAL {CHURN_DAYS} DAY''').df().T[0]
churn["churn_share"] = churn["churned"] / churn["eligible"]
print("Отток по правилу «нет покупок 90 дней» на конец данных:")
print(churn.to_string())

## 4. Метрики: пирамида

```
                    North Star: повторные покупатели в месяц
                  (покупали в месяце и в предыдущие 365 дней)
                    /                                    \
   Диагностическая 1:                          Диагностическая 2:
   конверсия новых во второй заказ             средний чек заказа, у.е.
   за 90 дней (приток в North Star)            (монетизация каждого визита)
                    \                                    /
              Риск-метрика: доля оттока 90 дней активной базы
```

| Уровень | Метрика | Зачем | Решение, которое на неё опирается |
|---|---|---|---|
| North Star | Повторные покупатели в месяц | Показывает ценность для покупателя: он нашёл подходящее и вернулся. Растёт вместе с бизнесом и не растёт от одних лишь трат на рекламу | Куда направлять продуктовые усилия: в удержание или в привлечение |
| Диагностическая | Конверсия во второй заказ за 90 дней | Основной вход в North Star: новый покупатель становится повторным. Сюда направлена гипотеза о рекомендациях | Запуск и оценка онбординга и персональных рекомендаций для новых; бюджет по каналам вместе с LTV/CAC |
| Диагностическая | Средний чек заказа | Следит, чтобы рост числа повторных шёл не за счёт мелких покупок; связывает North Star с выручкой и LTV | Кросс-селл в корзине, блок «дополните образ» |
| Риск | Доля оттока 90 дней среди активных в месяце | Ранний сигнал потери базы; защищает от решений, которые разово поднимают покупки (скидки), но ухудшают удержание | Стоп-критерий экспериментов, запуск реактивации |

Ниже — значения метрик по месяцам. Используются только полные месяцы и только месяцы, для которых хватает истории (365 дней до) и наблюдения после (90 дней).

In [ ]:
monthly = con.sql(f'''
    WITH o AS (
        SELECT customer_id, t_dat, rev,
               lag(t_dat)  OVER w AS prev_d,
               lead(t_dat) OVER w AS next_d,
               CAST(date_trunc('month', t_dat) AS DATE) AS m
        FROM orders WINDOW w AS (PARTITION BY customer_id ORDER BY t_dat)),
    bm AS (
        SELECT customer_id, m, COUNT(*) AS orders, SUM(rev) AS rev,
               bool_or(prev_d < m AND prev_d >= m - INTERVAL {RETURN_LOOKBACK_DAYS} DAY) AS is_ret,
               bool_or(next_d > last_day(m) AND next_d <= last_day(m) + INTERVAL {CHURN_DAYS} DAY) AS came_back
        FROM o GROUP BY 1, 2)
    SELECT m AS month, COUNT(*) AS active, SUM(CASE WHEN is_ret THEN 1 ELSE 0 END) AS "returning",
           SUM(orders) AS orders, SUM(rev) * {PRICE_SCALE} AS revenue,
           SUM(rev) * {PRICE_SCALE} / SUM(orders) AS aov,
           1 - AVG(CASE WHEN came_back THEN 1.0 ELSE 0 END) AS churn90
    FROM bm GROUP BY 1 ORDER BY 1''').df()
monthly["month"] = pd.to_datetime(monthly["month"])
# North Star считаем с сентября 2019: к этому моменту за каждым покупателем почти год истории
mstart = MAX_DATE.to_period("M").start_time
last_full_month = mstart if MAX_DATE == MAX_DATE + pd.offsets.MonthEnd(0) else mstart - pd.offsets.MonthBegin(1)
monthly.loc[monthly["month"] < COHORT_START, "returning"] = np.nan
monthly.loc[monthly["month"] > last_full_month, ["active", "returning", "orders", "revenue", "aov"]] = np.nan
churn_ok = monthly["month"].apply(lambda m: (m + pd.offsets.MonthEnd(0) + pd.Timedelta(days=CHURN_DAYS)) <= MAX_DATE)
monthly.loc[~churn_ok, "churn90"] = np.nan

conv = con.sql(f'''
    SELECT cohort AS month, COUNT(*) AS new_customers,
           AVG(CASE WHEN second_d <= first_d + INTERVAL {CONVERSION_DAYS} DAY THEN 1.0 ELSE 0 END) AS conv90
    FROM cust
    WHERE first_d >= DATE '{COHORT_START}'
      AND first_d <= DATE '{MAX_DATE.date()}' - INTERVAL {CONVERSION_DAYS} DAY
    GROUP BY 1 ORDER BY 1''').df()
conv["month"] = pd.to_datetime(conv["month"])
metrics = monthly.merge(conv, on="month", how="left")
show = metrics[metrics["month"] >= COHORT_START].copy()
show["month"] = show["month"].dt.strftime("%Y-%m")
print(show.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

In [ ]:
m = metrics.dropna(subset=["returning"])
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
axes[0, 0].plot(m["month"], m["returning"] / 1e3, marker="o", color="#4C72B0")
axes[0, 0].set_title("North Star: повторные покупатели, тыс.")
c = metrics.dropna(subset=["conv90"])
axes[0, 1].plot(c["month"], c["conv90"] * 100, marker="o", color="#55A868")
axes[0, 1].set_title("Конверсия во второй заказ за 90 дней, %")
axes[1, 0].plot(m["month"], m["aov"], marker="o", color="#8172B3")
axes[1, 0].set_title("Средний чек заказа, у.е.")
r = metrics.dropna(subset=["churn90"])
r = r[r["month"] >= m["month"].min()]
axes[1, 1].plot(r["month"], r["churn90"] * 100, marker="o", color="#C44E52")
axes[1, 1].set_title("Риск: отток 90 дней среди активных, %")
for ax in axes.flat:
    ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-05-31"), color="grey", alpha=0.15)
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Метрики пирамиды по месяцам (серым: закрытие магазинов, весна 2020)")
plt.tight_layout(); plt.show()

## 5. План трекинга

В данных один тип сырого события: строка покупки. Остальные события **восстанавливаются из транзакций** по определениям раздела 3: в живом продукте они отправлялись бы в трекинг отдельно, здесь каждое однозначно выводится из полей датасета. Для трека Б нужно минимум 3 события, описано 5.

In [ ]:
tracking = pd.DataFrame([
    ("item_purchased", "строка покупки в чеке",
     "customer_id, article_id, price, sales_channel_id, t_dat",
     "выручка, валовая прибыль, LTV", "покупка",
     "transactions_train: все поля; articles.garment_group_name"),
    ("order_completed", "все покупки покупателя за календарный день",
     "customer_id, t_dat, items, order_value, channel",
     "средний чек (диагн. 2), активные покупатели", "покупка",
     "transactions_train: t_dat, customer_id, price, sales_channel_id"),
    ("first_purchase", "первый order_completed покупателя",
     "customer_id, first_date, first_channel, age, club_member_status",
     "новые покупатели, когорта, CAC", "привлечение",
     "MIN(t_dat), sales_channel_id; customers.age, club_member_status"),
    ("repeat_purchase", "order_completed, перед которым был заказ этого покупателя",
     "customer_id, order_number, days_since_prev",
     "North Star, конверсия (диагн. 1), retention", "возврат",
     "t_dat (порядок и разница дат по customer_id)"),
    ("churned", "прошло 90 дней без order_completed",
     "customer_id, last_date, n_orders, ltv",
     "риск-метрика: доля оттока", "отток",
     "MAX(t_dat) по customer_id и дата конца данных"),
], columns=["событие", "триггер", "параметры", "метрика", "этап пути", "поле датасета"])
print(tracking.to_string(index=False))

In [ ]:
# Проверка: каждое событие выводится из данных (объёмы за весь период, без тестовых покупателей)
ev = con.sql(f'''
    SELECT
      (SELECT COUNT(*) FROM tx WHERE customer_id NOT IN (SELECT customer_id FROM test_users)) AS item_purchased,
      (SELECT COUNT(*) FROM orders) AS order_completed,
      (SELECT COUNT(*) FROM cust) AS first_purchase,
      (SELECT COUNT(*) FROM orders) - (SELECT COUNT(*) FROM cust) AS repeat_purchase,
      (SELECT COUNT(*) FROM cust WHERE last_d < DATE '{MAX_DATE.date()}' - INTERVAL {CHURN_DAYS} DAY) AS churned
''').df().T
ev.columns = ["событий"]
print(ev.to_string(formatters={"событий": "{:,}".format}))

## 6. Первые расчёты

### 6.1 Когортный retention

Когорты: месяц первого заказа, сентябрь 2019 — февраль 2020 (у каждой есть 6 полных месяцев наблюдения до сентября 2020).

In [ ]:
coh = con.sql(f'''
    SELECT c.cohort, c.first_ch,
           date_diff('month', c.cohort, CAST(date_trunc('month', o.t_dat) AS DATE)) AS k,
           COUNT(DISTINCT o.customer_id) AS active
    FROM orders o JOIN cust c USING (customer_id)
    WHERE c.cohort >= DATE '{COHORT_START}' AND c.cohort < DATE '{COHORT_END}'
    GROUP BY 1, 2, 3 HAVING k BETWEEN 0 AND {MAX_K}''').df()

act = coh.groupby(["cohort", "k"])["active"].sum().unstack("k")
sizes = act[0]
ret = act.div(sizes, axis=0)
ret.index = pd.to_datetime(ret.index).strftime("%Y-%m")
table = ret.copy()
table.insert(0, "размер", sizes.values)
print(table.to_string(float_format=lambda x: f"{x:.3f}"))

fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(ret.iloc[:, 1:].values * 100, cmap="Blues", aspect="auto", vmin=0)
ax.set_xticks(range(MAX_K)); ax.set_xticklabels([f"k={k}" for k in range(1, MAX_K + 1)])
ax.set_yticks(range(len(ret))); ax.set_yticklabels(ret.index)
for i in range(len(ret)):
    for j in range(MAX_K):
        ax.text(j, i, f"{ret.iloc[i, j + 1] * 100:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Retention по когортам, % покупателей с заказом в месяце k"); ax.grid(False)
plt.colorbar(im, ax=ax, fraction=0.03); plt.tight_layout(); plt.show()

In [ ]:
# Retention по каналу первой покупки (средневзвешенно по шести когортам)
by_ch = coh.groupby(["first_ch", "k"])["active"].sum().unstack("k")
ret_ch = by_ch.div(by_ch[0], axis=0)
print(ret_ch.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.2))
for ch, color in [(1, "#DD8452"), (2, "#4C72B0")]:
    ax.plot(range(1, MAX_K + 1), ret_ch.loc[ch, 1:] * 100, marker="o", color=color,
            label=f"канал {ch} (n={int(by_ch.loc[ch, 0]):,})")
ax.set_xlabel("месяц после первой покупки, k"); ax.set_ylabel("retention, %")
ax.set_title("Retention по каналу первой покупки"); ax.legend()
plt.tight_layout(); plt.show()

r1 = ret[1]
print(f"Retention месяца 1: от {r1.min():.1%} до {r1.max():.1%}; "
      f"месяц 3 в среднем {ret[3].mean():.1%}, месяц 6 - {ret[6].mean():.1%}")

**Выводы по retention.**
- Удержание низкое и быстро выходит на плато: в первый месяц после первой покупки возвращаются примерно 12–16 % новых покупателей, к третьему месяцу около 10 %, дальше уровень почти не меняется. Основная потеря происходит сразу после первой покупки, поэтому именно сюда направлена гипотеза о рекомендациях (раздел 1).
- Поздние когорты удерживаются хуже в первый месяц (у сентябрьской 16 %, у февральской около 12 %), часть разницы объясняется сезонностью: январь и февраль — слабые месяцы для покупок.
- Рост retention в месяцах 4–6 у зимних когорт (до 13 % у декабрьской) — календарный эффект летнего пика июня 2020, а не улучшение продукта. Когорты нужно сравнивать по одному календарному месяцу, а не только по возрасту.
- Retention различается по каналу первой покупки; эта разница реальная, а не сгенерированная, и учитывается в LTV ниже.

_Числа в выводах соответствуют полному прогону на данных соревнования; таблица выше пересчитывается при каждом запуске._

### 6.2 Маржа (сгенерировано)

В данных нет себестоимости, а LTV без маржи — это выручка, а не прибыль. Маржу генерируем по реальной структуре ассортимента:
1. для каждой из 21 группы одежды задаём **относительный уровень маржи** (базовые вещи и аксессуары выше, верхняя одежда и обувь ниже, раздел распродажи сильно ниже);
2. **калибруем** уровни так, чтобы средняя маржа, взвешенная по реальной выручке групп, совпала с публичной валовой маржой H&M Group за 2019 финансовый год — **52,6 %** (годовой отчёт H&M);
3. для каждого артикула добавляем разброс: бета-распределение вокруг маржи группы, стандартное отклонение 0,05, фиксированный seed.

In [ ]:
TARGET_GROSS_MARGIN = 0.526   # H&M Group, FY2019 (1.12.2018 - 30.11.2019), годовой отчёт
MARGIN_SD = 0.05
REL_MARGIN = {                # относительный уровень маржи группы (до калибровки)
    "Accessories": 1.15, "Socks and Tights": 1.12, "Jersey Basic": 1.10, "Under-, Nightwear": 1.08,
    "Jersey Fancy": 1.04, "Swimwear": 1.04, "Blouses": 1.02, "Dresses Ladies": 1.02, "Shirts": 1.00,
    "Skirts": 1.00, "Dressed": 0.98, "Knitwear": 0.98, "Shorts": 0.98, "Dresses/Skirts girls": 0.97,
    "Trousers": 0.95, "Woven/Jersey/Knitted mix Baby": 0.95, "Unknown": 0.95, "Trousers Denim": 0.92,
    "Outdoor": 0.88, "Shoes": 0.85, "Special Offers": 0.60,
}

grp = con.sql(f'''
    SELECT a.garment_group_name AS grp, COUNT(DISTINCT a.article_id) AS articles, SUM(t.price) AS rev
    FROM tx t JOIN articles a USING (article_id)
    WHERE t.t_dat >= DATE '{COHORT_START}' GROUP BY 1''').df()
print("Группы без заданного уровня (берётся 1.0):", sorted(set(grp["grp"]) - set(REL_MARGIN)) or "нет")
grp["rel"] = grp["grp"].map(REL_MARGIN).fillna(1.0)
grp["w"] = grp["rev"] / grp["rev"].sum()
scale = TARGET_GROSS_MARGIN / (grp["w"] * grp["rel"]).sum()
grp["margin"] = (grp["rel"] * scale).clip(0.05, 0.90)

rng = np.random.default_rng(SEED)
art = con.sql("SELECT article_id, garment_group_name AS grp FROM articles").df()
mean_m = art["grp"].map(grp.set_index("grp")["margin"]).fillna(TARGET_GROSS_MARGIN).values
kappa = mean_m * (1 - mean_m) / MARGIN_SD ** 2 - 1
art["margin"] = rng.beta(mean_m * kappa, (1 - mean_m) * kappa)
con.register("art_margin_df", art[["article_id", "margin"]])
con.execute("CREATE OR REPLACE TABLE art_margin AS SELECT * FROM art_margin_df")

check = con.sql(f'''
    SELECT SUM(t.price * m.margin) / SUM(t.price) FROM tx t JOIN art_margin m USING (article_id)
    WHERE t.t_dat >= DATE '{COHORT_START}' ''').fetchone()[0]
print(f"Средняя маржа, взвешенная по выручке, после генерации: {check:.1%} (цель {TARGET_GROSS_MARGIN:.1%})")
print(grp.sort_values("w", ascending=False)[["grp", "articles", "w", "margin"]]
      .rename(columns={"w": "доля выручки", "margin": "маржа"}).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

### 6.3 Расходы на привлечение и CAC (сгенерировано)

Расходов на маркетинг в данных нет. Генерируем дневную таблицу «дата × канал» так, чтобы она была связана с реальными данными:

`расход(день, канал) = новые покупатели канала в этот день (реальные) × CAC_BASE × сезонный множитель × шум`

- **CAC_BASE = 50 у.е.** одинаковый для обоих каналов, чтобы не вносить в данные разницу между каналами от себя: вся разница в LTV/CAC будет реальной (из LTV). Публичного CAC H&M нет, поэтому значение — сценарий, а основной результат — таблица чувствительности ниже;
- **сезонность:** в период Black Friday (22.11–02.12.2019) реклама дороже, множитель 1,3;
- **шум:** логнормальный, σ = 0,10 (ежедневные колебания аукционов и бюджетов), seed 42.

Канал здесь — канал первой покупки (канал продаж), а не рекламный канал привлечения; это ограничение допущения.

In [ ]:
CAC_BASE = {1: 50.0, 2: 50.0}
BF_START, BF_END, BF_MULT = "2019-11-22", "2019-12-02", 1.3
NOISE_SD = 0.10

spend = con.sql(f'''
    SELECT first_d AS date, first_ch AS ch, COUNT(*) AS new_customers
    FROM cust WHERE first_d >= DATE '{COHORT_START}' AND first_d < DATE '{COHORT_END}'
    GROUP BY 1, 2 ORDER BY 1, 2''').df()
spend["date"] = pd.to_datetime(spend["date"])
rng = np.random.default_rng(SEED)
mult = np.where(spend["date"].between(BF_START, BF_END), BF_MULT, 1.0)
noise = rng.lognormal(mean=-NOISE_SD ** 2 / 2, sigma=NOISE_SD, size=len(spend))
spend["spend"] = spend["new_customers"] * spend["ch"].map(CAC_BASE) * mult * noise
spend["is_synthetic"] = True
spend.to_csv(OUT_DIR / "hm_channel_daily_spend.csv", index=False)
print(f"Сохранено: {OUT_DIR / 'hm_channel_daily_spend.csv'} ({len(spend)} строк)")
print(spend.groupby("ch")[["new_customers", "spend"]].sum().assign(CAC=lambda d: d["spend"] / d["new_customers"]).round(2))

### 6.4 LTV и LTV/CAC

LTV — накопленная валовая прибыль на покупателя когорты за месяцы 0–6 (выручка × сгенерированная маржа товара). Это нижняя оценка: за 7 месяцев покупатель не исчерпывает свою ценность.

In [ ]:
ltv_raw = con.sql(f'''
    SELECT c.first_ch AS ch,
           date_diff('month', c.cohort, CAST(date_trunc('month', t.t_dat) AS DATE)) AS k,
           SUM(t.price) * {PRICE_SCALE} AS rev,
           SUM(t.price * COALESCE(m.margin, {TARGET_GROSS_MARGIN})) * {PRICE_SCALE} AS gp
    FROM tx t JOIN cust c USING (customer_id) LEFT JOIN art_margin m USING (article_id)
    WHERE c.cohort >= DATE '{COHORT_START}' AND c.cohort < DATE '{COHORT_END}'
    GROUP BY 1, 2 HAVING k BETWEEN 0 AND {MAX_K}''').df()

n_new = spend.groupby("ch")["new_customers"].sum()
rev_cum = ltv_raw.pivot(index="ch", columns="k", values="rev").fillna(0).cumsum(axis=1).div(n_new, axis=0)
gp_cum = ltv_raw.pivot(index="ch", columns="k", values="gp").fillna(0).cumsum(axis=1).div(n_new, axis=0)
print("LTV по выручке, у.е. на покупателя (накопленно):"); print(rev_cum.round(1).to_string())
print("\nLTV по валовой прибыли, у.е. на покупателя (накопленно):"); print(gp_cum.round(1).to_string())

cac = spend.groupby("ch")["spend"].sum() / n_new
res = pd.DataFrame({
    "новых покупателей": n_new,
    "расход, у.е.": spend.groupby("ch")["spend"].sum().round(0),
    "CAC": cac.round(2),
    "LTV выручка 0-6": rev_cum[MAX_K].round(2),
    "LTV прибыль 0-6": gp_cum[MAX_K].round(2),
    "LTV/CAC": (gp_cum[MAX_K] / cac).round(2),
    "окупаемость, месяц": [next((k for k in gp_cum.columns if gp_cum.loc[ch, k] >= cac[ch]), None) for ch in gp_cum.index],
    "безубыточный CAC": gp_cum[MAX_K].round(2),
})
res.index.name = "канал"
print("\n", res.to_string())
res.to_csv(OUT_DIR / "hm_ltv_cac_by_channel.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for ch, color in [(1, "#DD8452"), (2, "#4C72B0")]:
    ax.plot(gp_cum.columns, gp_cum.loc[ch], marker="o", color=color, label=f"LTV, канал {ch}")
    ax.axhline(cac[ch], color=color, ls="--", lw=1)
ax.text(MAX_K, cac.mean() * 1.03, "CAC (сценарий)", ha="right", fontsize=8)
ax.set_xlabel("месяц после первой покупки, k"); ax.set_ylabel("у.е. на покупателя")
ax.set_title("Накопленная валовая прибыль на покупателя против CAC"); ax.legend()
plt.tight_layout(); plt.show()

# Чувствительность: LTV/CAC при других CAC и сдвиге маржи на +-10 п.п.
rows = []
for cac_v in [20, 35, 50, 65, 80, 100]:
    for dm in [-0.10, 0.0, 0.10]:
        row = {"CAC": cac_v, "сдвиг маржи, п.п.": int(round(dm * 100))}
        for ch in gp_cum.index:
            ltv_v = gp_cum.loc[ch, MAX_K] + dm * rev_cum.loc[ch, MAX_K]
            row[f"LTV/CAC канал {ch}"] = round(ltv_v / cac_v, 2)
        rows.append(row)
sens = pd.DataFrame(rows)
print(sens.pivot(index="CAC", columns="сдвиг маржи, п.п.").round(2).to_string(float_format=lambda x: f"{x:.2f}"))

**Выводы по LTV/CAC.**
- Реальная часть результата — LTV. Покупатель канала 2 приносит за 7 месяцев примерно на четверть больше выручки и прибыли, чем покупатель канала 1, и разница складывается в основном из размера первого заказа.
- **Безубыточный CAC** (сколько можно потратить на привлечение, чтобы окупиться за месяцы 0–6) — около 83 у.е. для канала 1 и 103 у.е. для канала 2. Этот вывод не зависит от сгенерированного CAC.
- В центральном сценарии (CAC около 50 у.е.) LTV/CAC около 1,6 для канала 1 и 2,0 для канала 2, привлечение окупается примерно за 3 месяца и за месяц соответственно. По таблице чувствительности привлечение остаётся окупаемым при CAC до 80 у.е. в канале 1 и до 100 у.е. в канале 2; сдвиг маржи на ±10 п.п. меняет LTV/CAC примерно на ±0,3–0,4.
- Значения LTV/CAC — сценарий, а не факт: они верны только при допущении о CAC. Для продуктовых решений используем безубыточный CAC и относительное сравнение каналов.

## 7. Этика (краткий комментарий, трек Б)

- **Цель.** Учебный анализ поведения покупателей для выбора продуктовых метрик; никаких решений в отношении конкретных людей не принимается.
- **Происхождение и согласие.** Данные опубликованы самой H&M Group для соревнования Kaggle; сбор и согласие покупателей — на стороне H&M. Используем их по правилам соревнования (некоммерческое академическое использование) и **не распространяем**: в репозитории только код и агрегированные результаты.
- **Персональные данные.** Идентификаторы покупателей и почтовые индексы захешированы. Специальных категорий данных (здоровье, религия, политика и т. п.) нет. Квазиидентификаторы — возраст, хеш индекса и история покупок; их сочетание теоретически позволяет повторную идентификацию. Меры: почтовый индекс не используем, результаты публикуем только агрегатами по группам из сотен и тысяч покупателей, попыток деанонимизации не делаем.
- **Передача третьим лицам** — нет. **Хранение и удаление:** файлы лежат локально в `data/` (исключена из git) и удаляются после окончания курса; токен Kaggle не хранится в коде.
- **Риск неравного отношения.** Сегментация по возрасту может привести к решениям, которые хуже обслуживают отдельные возрастные группы. Возраст используем только для описания аудитории и диагностики, не для таргетинга.

## 8. Критерии успеха

Базовый уровень берём по докризисным когортам и месяцам (сентябрь 2019 — февраль 2020), чтобы пандемия не искажала точку отсчёта. Целевые значения относятся к гипотезе раздела 1 (персональные рекомендации для новых покупателей), которая проверяется в следующих сдачах.

In [ ]:
base_mask = (metrics["month"] >= COHORT_START) & (metrics["month"] < COHORT_END)
b = metrics[base_mask]
conv_base = (b["conv90"] * b["new_customers"]).sum() / b["new_customers"].sum()
nsm_base = b["returning"].mean()
churn_base = b["churn90"].mean()

# Минимально обнаружимый эффект для конверсии: месячная когорта новых делится 50/50, alpha = 0.05, мощность 0.8
z_a, z_b = 1.96, 0.84
n_group = b["new_customers"].mean() / 2
mde = (z_a + z_b) * np.sqrt(2 * conv_base * (1 - conv_base) / n_group)

criteria = pd.DataFrame([
    ("North Star: повторные покупатели в месяц", f"{nsm_base:,.0f}", f"+3 % ({nsm_base * 1.03:,.0f})",
     "сравнение с тем же месяцем прошлого года и контрольной группой; бутстреп-интервал разницы"),
    ("Конверсия во второй заказ за 90 дней", f"{conv_base:.1%}", f"+2 п.п. ({conv_base + 0.02:.1%})",
     f"A/B-тест на новых покупателях, z-тест пропорций, alpha=0,05, мощность 0,8; MDE на одной месячной когорте {mde * 100:.1f} п.п."),
    ("Риск: отток 90 дней активной базы", f"{churn_base:.1%}", f"не выше {churn_base + 0.01:.1%} (+1 п.п.)",
     "стоп-критерий эксперимента: если отток растёт больше чем на 1 п.п., изменение не принимается"),
], columns=["метрика", "база (сен 2019 - фев 2020)", "цель к финалу", "способ проверки"])
print(criteria.to_string(index=False))

Цель по конверсии (+2 п.п.) выбрана так, чтобы она была больше минимально обнаружимого эффекта на одной месячной когорте: тогда эксперимент на реалистичном объёме даст однозначный ответ. Средний чек (диагностическая 2) отслеживается как контрольная метрика: допустимое снижение не больше 3 %.

## 9. Ограничения и обоснование синтетических данных

**Ограничения датасета**
- Единственное сырое событие — покупка; нет просмотров, корзины, возвратов и времени суток. Воронку до покупки построить нельзя, конверсия считается в повторную покупку.
- Нет даты регистрации: дата появления — первая покупка, нужен прогрев истории (когорты с сентября 2019).
- Нет устройства, гео только в виде хеша индекса (8,8 % заглушка). Срезы: канал продаж, возраст, статус клуба, группа товаров.
- Канал продаж 1 и 2 без расшифровки; это не рекламный канал привлечения.
- Цена нормирована, все деньги в условных единицах.
- Весна 2020 (пандемия) — структурный сдвиг; базовые уровни берём до марта 2020.

**Синтетические данные: что, зачем и как**

Синтетика добавлена только там, где без неё нельзя посчитать метрику из задания (согласовано с преподавателем).

| Что | Без него не посчитать | Как сгенерировано и к чему привязано | Что остаётся реальным |
|---|---|---|---|
| Маржа артикула | LTV по прибыли, LTV/CAC | Относительные уровни по 21 реальной группе одежды, калибровка к валовой марже H&M за 2019 год (52,6 %), бета-шум σ=0,05, seed 42 | Выручка, структура ассортимента, доли групп |
| Расходы на привлечение по дням и каналам | CAC, LTV/CAC, окупаемость | Реальное число новых покупателей канала в день × CAC 50 у.е. × множитель Black Friday 1,3 × логнормальный шум σ=0,10, seed 42; файл `outputs/hm_channel_daily_spend.csv` с флагом `is_synthetic` | Число новых покупателей, сезонность, LTV и безубыточный CAC |

Как допущения влияют на выводы, показывает таблица чувствительности в 6.4. Доля сгенерированного: одна вспомогательная таблица расходов и один столбец маржи; транзакции, покупатели и товары не изменялись. Синтетические дефекты не добавлялись.

**Вклад:** работа выполнена одним автором (трек Б).